<a href="https://colab.research.google.com/github/Solmaeir/Roman_Columns/blob/main/Roman_Columns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Roma Dönemi Kolon Restorasyonu
**Görüntü İşleme Projesi — v3**

### Pipeline
1. Bilateral Filter + CLAHE ön işleme  
2. SAM segmentasyonu (→ Klasik CV fallback)  
3. **Hasar yönü analizi**: Alt/üst kesiği mi? Sol/sağ kopmalar mı?  
4. **Dikey uzatma**: Şaft dokusunu tile + LaMa/TELEA ile pürüzsüzleştir  
5. **Yatay simetri**: Yalnızca sol/sağ hasar varsa uygula  
6. Önce / Sonra görselleştirme

In [ ]:
# ── Google Drive bağla + klasör yolları ──────────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB      = True
    BASE          = '/content/drive/MyDrive/sutun_veriseti_1'
    DAMAGED_DIR   = os.path.join(BASE, 'damaged_column')
    REFERENCE_DIR = os.path.join(BASE, 'referance_column')
    print('Google Colab — Drive bağlandı.')
except ImportError:
    IN_COLAB      = False
    BASE          = '.'
    DAMAGED_DIR   = os.path.join(BASE, 'damaged_column')
    REFERENCE_DIR = os.path.join(BASE, 'referance_column')
    print('Yerel ortam.')

damaged_files   = [f for f in os.listdir(DAMAGED_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
reference_files = [f for f in os.listdir(REFERENCE_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f'Damaged  : {len(damaged_files)} görüntü')
print(f'Reference: {len(reference_files)} görüntü')
print(f'Test [25]: {damaged_files[25]}')

In [ ]:
# ── Bağımlılıklar (sürüm çakışması yok) ─────────────────────────────────────
import subprocess, sys

def pip_q(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

try:
    import segment_anything; print('segment-anything OK')
except ImportError:
    pip_q('segment-anything')

try:
    import simple_lama_inpainting; print('simple-lama-inpainting OK')
except ImportError:
    pip_q('simple-lama-inpainting')

CKPT = 'sam_vit_b_01ec64.pth'
if not os.path.exists(CKPT):
    print('SAM checkpoint indiriliyor…')
    subprocess.run(['wget', '-q',
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'], check=False)
    print('OK')
else:
    print('SAM checkpoint mevcut.')

In [ ]:
# ── Import'lar ───────────────────────────────────────────────────────────────
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')
print(f'Torch  : {torch.__version__}')
print(f'PIL    : {Image.__version__}')

SAM_AVAILABLE = False
try:
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    SAM_AVAILABLE = True
    print(f'SAM    : OK  (CUDA={torch.cuda.is_available()})')
except Exception as e:
    print(f'SAM    : Yok — {e}')

LAMA_AVAILABLE = False
try:
    from simple_lama_inpainting import SimpleLama
    LAMA_AVAILABLE = True
    print('LaMa   : OK')
except Exception as e:
    print(f'LaMa   : Yok — {e}')

In [ ]:
# ── Modelleri yükle ──────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

mask_generator = None
if SAM_AVAILABLE:
    try:
        _sam = sam_model_registry['vit_b'](checkpoint=CKPT)
        _sam.to(DEVICE)
        mask_generator = SamAutomaticMaskGenerator(
            model=_sam,
            points_per_side=32,
            pred_iou_thresh=0.86,
            stability_score_thresh=0.90,
            min_mask_region_area=300,
        )
        print(f'SAM yüklendi ({DEVICE}).')
    except Exception as e:
        print(f'SAM yüklenemedi: {e}')
        SAM_AVAILABLE = False

lama_model = None
if LAMA_AVAILABLE:
    try:
        lama_model = SimpleLama()
        print('LaMa yüklendi.')
    except Exception as e:
        print(f'LaMa yüklenemedi: {e}')
        LAMA_AVAILABLE = False

if not SAM_AVAILABLE:   print('→ Klasik CV aktif.')
if not LAMA_AVAILABLE:  print('→ OpenCV TELEA aktif.')

In [ ]:
# ── Yardımcı fonksiyonlar ─────────────────────────────────────────────────────

def preprocess(img_bgr):
    """Bilateral + CLAHE ile kontrast iyileştir."""
    dn = cv2.bilateralFilter(img_bgr, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(dn, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)


def column_score(cnt, h_img, w_img):
    area = cv2.contourArea(cnt)
    if area < 500: return 0.0
    x, y, w, h = cv2.boundingRect(cnt)
    asp = h / (w + 1e-5)
    ar  = area / (h_img * w_img)
    return asp * ar if asp > 1.5 and 0.03 < ar < 0.80 else 0.0


def detect_column_classical(img_bgr):
    h_img, w_img = img_bgr.shape[:2]
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    adapt   = cv2.adaptiveThreshold(blurred, 255,
                  cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 5)
    edges   = cv2.Canny(blurred, 20, 80)
    combo   = cv2.bitwise_or(adapt, edges)
    kr = cv2.getStructuringElement(cv2.MORPH_RECT,   (5, 5))
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9, 9))
    closed  = cv2.morphologyEx(combo,   cv2.MORPH_CLOSE, kr, iterations=4)
    dilated = cv2.dilate(closed, kr, iterations=2)
    cnts, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    scored = sorted(cnts, key=lambda c: column_score(c, h_img, w_img), reverse=True)
    best   = scored[0]
    mask   = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.drawContours(mask, [best], -1, 255, -1)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, ke, iterations=5)


def detect_column_sam(img_bgr):
    if mask_generator is None: return None
    h_img, w_img = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    try:
        masks = mask_generator.generate(img_rgb)
    except Exception as e:
        print(f'  SAM hatası: {e}'); return None
    if not masks: return None
    ms = sorted(masks, key=lambda m: m['area'], reverse=True)
    best, best_s = None, 0.0
    for m in ms[:12]:
        seg = m['segmentation'].astype(np.uint8) * 255
        cnts, _ = cv2.findContours(seg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: continue
        s = column_score(max(cnts, key=cv2.contourArea), h_img, w_img)
        if s > best_s: best_s, best = s, seg
    if best is None:
        best = ms[0]['segmentation'].astype(np.uint8) * 255
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    return cv2.morphologyEx(best, cv2.MORPH_CLOSE, ke, iterations=4)


def detect_column(img_bgr):
    if SAM_AVAILABLE:
        m = detect_column_sam(img_bgr)
        if m is not None:
            print('  SAM ile tespit edildi.'); return m, 'SAM'
    m = detect_column_classical(img_bgr)
    if m is not None:
        print('  Klasik CV ile tespit edildi.'); return m, 'Klasik CV'
    return None, None


def get_bbox(mask):
    """Maskenin bounding box ve simetri eksenini döndür."""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    cnt = max(cnts, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(cnt)
    return x, y, w, h, x + w // 2, cnt


print('Yardımcı fonksiyonlar hazır.')

In [ ]:
# ── Hasar yönü analizi ────────────────────────────────────────────────────────

def analyze_damage(col_mask):
    """
    Kolonun 4 yönündeki hasar oranını döndür (0-1).
    Yüksek değer = o yönde çok hasar var.

    Yöntem:
    - İdeal (dolu) bounding box ile gerçek maskeyi karşılaştır.
    - Her çeyrek bölgedeki eksik piksel oranı = hasar skoru.
    """
    info = get_bbox(col_mask)
    if info is None:
        return {'top': 0, 'bottom': 0, 'left': 0, 'right': 0}
    x, y, w, h, cx, _ = info

    h_img, w_img = col_mask.shape[:2]
    mid_y = y + h // 2

    # İdeal dikdörtgen (dolu, hasarsız kolon)
    ideal = np.zeros_like(col_mask)
    ideal[y:y+h, x:x+w] = 255

    def ratio(region_ideal, region_actual):
        denom = np.sum(region_ideal > 0)
        if denom == 0: return 0.0
        return max(0.0, (denom - np.sum(region_actual > 0)) / denom)

    top_dmg    = ratio(ideal[:mid_y,    x:x+w], col_mask[:mid_y,    x:x+w])
    bottom_dmg = ratio(ideal[mid_y:y+h, x:x+w], col_mask[mid_y:y+h, x:x+w])
    left_dmg   = ratio(ideal[y:y+h,     x:cx],  col_mask[y:y+h,     x:cx])
    right_dmg  = ratio(ideal[y:y+h,     cx:x+w],col_mask[y:y+h,     cx:x+w])

    # Alt kısım ayrıca: kolonun alt sınırı ile görüntü alt sınırı arasındaki boşluk
    gap_bottom = (h_img - (y + h)) / h_img  # görüntünün ne kadarı boş altta

    d = {'top': top_dmg, 'bottom': bottom_dmg,
         'left': left_dmg, 'right': right_dmg,
         'gap_bottom': gap_bottom}

    print(f'  Hasar analizi: Üst={top_dmg:.2f}  Alt={bottom_dmg:.2f}  '
          f'Sol={left_dmg:.2f}  Sağ={right_dmg:.2f}  '
          f'Alt-boşluk={gap_bottom:.2f}')
    return d


print('analyze_damage() hazır.')

In [ ]:
# ── Dikey uzatma: şaft dokusunu aşağıya / yukarıya uzat ─────────────────────

def _make_seamless_strip(strip):
    """
    Şerit tekrarında dikişsiz geçiş için gradyan harmanla.
    strip: (H, W, 3) uint8
    """
    h = strip.shape[0]
    # Üst ve alt kenarları karıştır
    blend_h = max(4, h // 6)
    result  = strip.astype(np.float32)
    for i in range(blend_h):
        alpha = i / blend_h
        result[i]      = result[i]      * alpha + strip[-blend_h + i] * (1 - alpha)
        result[-i - 1] = result[-i - 1] * alpha + strip[blend_h - i - 1] * (1 - alpha)
    return np.clip(result, 0, 255).astype(np.uint8)


def extend_column_down(img_bgr, col_mask, extra_ratio=0.45):
    """
    Kolon şaftını aşağı uzat:
    1. Şaft'ın orta bölümünden doku şeridi al (capital + kırık kenar hariç)
    2. Şeridi döşe (seamless)
    3. Kenar inpainting ile seam gizle

    extra_ratio : Mevcut yüksekliğin kaçı kadar uzat (0.45 = %45)
    Döndürür: (uzatılmış_bgr, uzatma_maskesi)
    """
    h_img, w_img = img_bgr.shape[:2]
    info = get_bbox(col_mask)
    if info is None:
        return img_bgr.copy(), np.zeros((h_img, w_img), dtype=np.uint8)

    x, y, w, h, cx, _ = info
    col_bottom = y + h

    # Ne kadar uzatacağız?
    extend_px    = int(h * extra_ratio)
    target_bot   = min(h_img - 5, col_bottom + extend_px)

    if target_bot <= col_bottom + 10:
        print('  Dikey uzatma gerekmiyor.')
        return img_bgr.copy(), np.zeros((h_img, w_img), dtype=np.uint8)

    print(f'  Alt uzatma: {col_bottom}→{target_bot} px  (+{target_bot-col_bottom} px)')

    # ── Doku şeridi: şaft ortası (%30-%75 arası yükseklik) ──
    tex_y0 = y + int(h * 0.30)   # capital sonrası
    tex_y1 = y + int(h * 0.75)   # alt hasar başlamadan önce
    if tex_y1 - tex_y0 < 20:
        tex_y0 = y + int(h * 0.35)
        tex_y1 = y + int(h * 0.65)

    texture = img_bgr[tex_y0:tex_y1, x:x + w].copy()
    texture = _make_seamless_strip(texture)
    tex_h   = texture.shape[0]

    # ── Döşeme: uzatma bölgesini doku ile doldur ──
    result   = img_bgr.copy()
    ext_mask = np.zeros((h_img, w_img), dtype=np.uint8)

    fill_y = col_bottom
    while fill_y < target_bot:
        seg_end = min(fill_y + tex_h, target_bot)
        seg_h   = seg_end - fill_y
        # Genişliği şaftın genişliğine kısıtla
        seg_w   = min(w, w_img - x)
        result[fill_y:seg_end, x:x + seg_w] = texture[:seg_h, :seg_w]
        ext_mask[fill_y:seg_end, x:x + seg_w] = 255
        fill_y += tex_h

    # ── Geçiş bölgesi: mevcut kolon alt kenarı + uzatma başlangıcı ──
    # Burayı inpainting ile yumuşat
    seam_h    = min(30, extend_px // 4)
    seam_mask = np.zeros((h_img, w_img), dtype=np.uint8)
    sy0 = max(0, col_bottom - seam_h // 2)
    sy1 = min(h_img, col_bottom + seam_h // 2)
    seam_mask[sy0:sy1, x:x + w] = 255

    return result, ext_mask, seam_mask


def extend_column_up(img_bgr, col_mask, extra_ratio=0.25):
    """Kolon şaftını yukarı uzat (alt uzatmanın simetriği)."""
    h_img, w_img = img_bgr.shape[:2]
    info = get_bbox(col_mask)
    if info is None:
        return img_bgr.copy(), np.zeros((h_img, w_img), dtype=np.uint8)

    x, y, w, h, cx, _ = info

    extend_px  = int(h * extra_ratio)
    target_top = max(5, y - extend_px)
    if y - target_top < 10:
        return img_bgr.copy(), np.zeros((h_img, w_img), dtype=np.uint8)

    tex_y0  = y + int(h * 0.25)
    tex_y1  = y + int(h * 0.60)
    texture = img_bgr[tex_y0:tex_y1, x:x + w].copy()
    texture = cv2.flip(_make_seamless_strip(texture), 0)  # yukarı için ters çevir
    tex_h   = texture.shape[0]

    result   = img_bgr.copy()
    ext_mask = np.zeros((h_img, w_img), dtype=np.uint8)

    fill_y = y
    while fill_y > target_top:
        seg_start = max(fill_y - tex_h, target_top)
        seg_h     = fill_y - seg_start
        result[seg_start:fill_y, x:x + w] = texture[-seg_h:, :]
        ext_mask[seg_start:fill_y, x:x + w] = 255
        fill_y -= tex_h

    seam_h    = min(30, extend_px // 4)
    seam_mask = np.zeros((h_img, w_img), dtype=np.uint8)
    seam_mask[max(0,y-seam_h//2):min(h_img,y+seam_h//2), x:x+w] = 255

    return result, ext_mask, seam_mask


print('Dikey uzatma fonksiyonları hazır.')

In [ ]:
# ── Yatay simetri restorasyonu (sol/sağ hasar) ───────────────────────────────

def symmetry_restore(img_bgr, col_mask):
    """
    Sol/sağ simetri:
    - Daha dolu olan tarafı aynalayarak eksik tarafa uygula.
    - Yalnızca kolon maskesinin boş olduğu pikselleri doldur.
    """
    h_img, w_img = img_bgr.shape[:2]
    info = get_bbox(col_mask)
    if info is None:
        return img_bgr.copy(), None

    x, y, w, h, cx, _ = info

    left_px  = np.sum(col_mask[:, x:cx]    > 128)
    right_px = np.sum(col_mask[:, cx:x + w] > 128)

    result = img_bgr.copy()

    if left_px >= right_px:
        good   = img_bgr[:, x:cx].copy()
        mirror = cv2.flip(good, 1)
        mw     = mirror.shape[1]
        xs, xe = cx, min(w_img, cx + mw)
        aw     = xe - xs
        empty  = col_mask[:, xs:xe] == 0
        tmp    = result[:, xs:xe].copy()
        tmp[empty] = mirror[:, :aw][empty]
        result[:, xs:xe] = tmp
        side = 'sağ (sol referans)'
    else:
        good   = img_bgr[:, cx:x + w].copy()
        mirror = cv2.flip(good, 1)
        mw     = mirror.shape[1]
        xs, xe = max(0, cx - mw), cx
        aw     = xe - xs
        empty  = col_mask[:, xs:xe] == 0
        tmp    = result[:, xs:xe].copy()
        tmp[empty] = mirror[:, mw - aw:][empty]
        result[:, xs:xe] = tmp
        side = 'sol (sağ referans)'

    print(f'  Simetri: {side} | Sol={left_px}px Sağ={right_px}px')
    return result, cx


print('symmetry_restore() hazır.')

In [ ]:
# ── İnpainting: LaMa → OpenCV TELEA fallback ────────────────────────────────

def inpaint(img_bgr, mask):
    """mask=255 olan bölgeleri LaMa veya TELEA ile doldur."""
    if np.sum(mask) < 50:
        return img_bgr.copy()

    if LAMA_AVAILABLE and lama_model is not None:
        try:
            img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            res_pil  = lama_model(Image.fromarray(img_rgb), Image.fromarray(mask))
            return cv2.cvtColor(np.array(res_pil), cv2.COLOR_RGB2BGR)
        except Exception as e:
            print(f'  LaMa hatası: {e} → TELEA')

    return cv2.inpaint(img_bgr, mask, inpaintRadius=7, flags=cv2.INPAINT_TELEA)


print('inpaint() hazır.')

In [ ]:
# ── Ana Pipeline ─────────────────────────────────────────────────────────────

def restore_column(
    img_path,
    extra_bottom=0.45,   # Mevcut yüksekliğin %45'i kadar alta uzat
    extra_top=0.20,      # Mevcut yüksekliğin %20'si kadar üste uzat
    sym_threshold=0.15,  # Yatay hasar bu eşiği geçerse simetri uygula
    bottom_gap_thresh=0.08,  # Alt boşluk bu değerin üzerindeyse uzatma yap
):
    """
    Akıllı restorasyon pipeline:
    1. Kolon segmente et
    2. Hasar yönünü analiz et
    3. Dikey uzatma (alt/üst boşluk varsa)
    4. Yatay simetri (sol/sağ hasar eşiği aşılırsa)
    5. Seam inpainting
    6. Son filtre + görselleştir
    """
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f'Görüntü okunamadı: {img_path}')

    fname = os.path.basename(img_path)
    H, W  = img.shape[:2]
    print(f"\n{'='*65}")
    print(f'  {fname}  ({W}×{H} px)')
    print(f"{'='*65}")

    # 1. Ön işleme
    print('① Ön işleme…')
    proc = preprocess(img)

    # 2. Segmentasyon
    print('② Segmentasyon…')
    col_mask, method = detect_column(proc)
    if col_mask is None:
        print('HATA: Kolon tespit edilemedi.'); return None

    # 3. Hasar analizi
    print('③ Hasar analizi…')
    dmg = analyze_damage(col_mask)
    info = get_bbox(col_mask)
    x, y, w, h, cx, _ = info

    current = proc.copy()
    all_masks = []
    steps_log = []

    # 4a. ALT UZATMA
    # Koşul: görüntünün altında yeterince boşluk var
    #        VEYA alt hasar oranı yüksek
    do_bottom = dmg['gap_bottom'] > bottom_gap_thresh or dmg['bottom'] > 0.25
    if do_bottom:
        print('④ Alt uzatma uygulanıyor…')
        out = extend_column_down(current, col_mask, extra_ratio=extra_bottom)
        ext_bgr, ext_mask, seam_mask = out
        # Seam bölgesini inpaint
        ext_bgr = inpaint(ext_bgr, seam_mask)
        current = ext_bgr
        all_masks.append(('Alt Uzatma', ext_mask))
        steps_log.append('alt_uzatma')
    else:
        print('④ Alt uzatma gerekmedi.')

    # 4b. ÜST UZATMA
    do_top = dmg['top'] > 0.25
    if do_top:
        print('⑤ Üst uzatma uygulanıyor…')
        out = extend_column_up(current, col_mask, extra_ratio=extra_top)
        ext_bgr, ext_mask, seam_mask = out
        ext_bgr = inpaint(ext_bgr, seam_mask)
        current = ext_bgr
        all_masks.append(('Üst Uzatma', ext_mask))
        steps_log.append('ust_uzatma')

    # 4c. YATAY SİMETRİ
    do_sym = max(dmg['left'], dmg['right']) > sym_threshold
    if do_sym:
        print('⑥ Yatay simetri uygulanıyor…')
        sym_bgr, sym_axis = symmetry_restore(current, col_mask)
        # İdeal mask ile eksik bölgeyi hesapla
        ideal = np.zeros_like(col_mask)
        ideal[y:y+h, x:x+w] = 255
        ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
        col_f   = cv2.morphologyEx(col_mask, cv2.MORPH_CLOSE, ke, iterations=3)
        sym_gap = cv2.bitwise_and(ideal, cv2.bitwise_not(col_f))
        sym_gap = cv2.morphologyEx(sym_gap, cv2.MORPH_OPEN, ke, iterations=1)
        if np.sum(sym_gap) > 200:
            sym_bgr = inpaint(sym_bgr, sym_gap)
        current = sym_bgr
        all_masks.append(('Simetri Boşluğu', sym_gap))
        steps_log.append('simetri')
    else:
        print('⑥ Yatay hasar eşiğin altında, simetri atlandı.')

    if not steps_log:
        print('  ⚠ Hiçbir restorasyon adımı tetiklenmedi!')
        print('  İpucu: extra_bottom veya sym_threshold parametrelerini düşür.')

    # 5. Son bilateral filtre
    final_bgr = cv2.bilateralFilter(current, d=7, sigmaColor=50, sigmaSpace=50)

    # ── RGB'ye çevir ──
    img_rgb   = cv2.cvtColor(img,       cv2.COLOR_BGR2RGB)
    proc_rgb  = cv2.cvtColor(proc,      cv2.COLOR_BGR2RGB)
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)

    # Simetri ekseni görseli
    vis_mask = img_rgb.copy()
    boundary = cv2.Canny(col_mask, 50, 150)
    vis_mask[boundary > 0] = [50, 255, 50]
    if do_sym:
        cv2.line(vis_mask, (cx, 0), (cx, H), (255, 50, 50), 2)

    # Tüm maskelerin birleşimi
    combined_mask = np.zeros((H, W), dtype=np.uint8)
    for _, m in all_masks:
        combined_mask = cv2.bitwise_or(combined_mask, m)

    # ── Görselleştirme ──
    fig, axes = plt.subplots(1, 5, figsize=(26, 8))

    axes[0].imshow(img_rgb)
    axes[0].set_title('① Orijinal (Hasarlı)', fontsize=12, fontweight='bold', color='darkred')

    axes[1].imshow(proc_rgb)
    axes[1].set_title(f'② Ön İşlenmiş\nBilateral+CLAHE', fontsize=11)

    axes[2].imshow(col_mask, cmap='gray')
    axes[2].set_title(f'③ Kolon Maskesi\n({method})', fontsize=11)

    axes[3].imshow(combined_mask, cmap='hot')
    axes[3].set_title('④ Uygulanan Bölge\n(sarı=dolduruldu)', fontsize=11)

    axes[4].imshow(final_rgb)
    axes[4].set_title('⑤ RESTORE EDİLMİŞ', fontsize=12,
                      fontweight='bold', color='darkgreen')

    for ax in axes: ax.axis('off')
    methods_str = ' + '.join(steps_log) if steps_log else 'yok'
    plt.suptitle(f'Roma Kolon Restorasyonu — {fname}\n[{methods_str}]',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('restoration_steps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("→ 'restoration_steps.png' kaydedildi.")

    return {'original': img_rgb, 'final': final_rgb,
            'col_mask': col_mask, 'combined_mask': combined_mask,
            'damage': dmg, 'steps': steps_log}


print('restore_column() hazır.')

In [ ]:
# ── TEST: damaged_files[25] ───────────────────────────────────────────────────
# a9ca5f5b6ba9cd36e161e86f0fa4925b.jpg

damaged_files = [f for f in os.listdir(DAMAGED_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

test_file = damaged_files[25]
test_path = os.path.join(DAMAGED_DIR, test_file)
print(f'Test: [{25}] {test_file}')

result = restore_column(
    test_path,
    extra_bottom=0.50,       # Şaftı %50 uzat (resme göre artırılabilir)
    extra_top=0.15,
    sym_threshold=0.12,      # Düşük eşik = küçük sol/sağ hasarı da yakala
    bottom_gap_thresh=0.05,  # Çok küçük eşik = neredeyse her zaman alt uzatma uygula
)

In [ ]:
# ── Önce / Sonra büyük karşılaştırma ─────────────────────────────────────────

if result is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 14))

    ax1.imshow(result['original'])
    ax1.set_title('ÖNCE\n(Hasarlı Kolon)', fontsize=18, fontweight='bold', color='darkred')
    ax1.axis('off')

    ax2.imshow(result['final'])
    ax2.set_title('SONRA\n(Restore Edilmiş)', fontsize=18, fontweight='bold', color='darkgreen')
    ax2.axis('off')

    plt.suptitle('Roma Kolon Dijital Restorasyonu', fontsize=20, fontweight='bold')
    plt.tight_layout()
    plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("→ 'before_after.png' kaydedildi.")
    print(f"Uygulanan adımlar: {result['steps']}")

In [ ]:
# ── Parametre ayarı: Uzatma miktarını görsel olarak test et ──────────────────
# Sonuç istediğin gibi değilse bu hücreyi değiştirerek dene.

# extra_bottom : Daha büyük → daha uzun şaft
# extra_top    : Başlık üstüne uzatma (çoğu zaman 0)
# sym_threshold: 0 = her zaman simetri, 1 = asla

# Örnek: çok kısa çıktıysa
# result = restore_column(test_path, extra_bottom=0.80, bottom_gap_thresh=0.01)

# Örnek: simetri de uygulanmasını istiyorsan
# result = restore_column(test_path, extra_bottom=0.50, sym_threshold=0.05)

print('Parametre ayarı için yorumları kaldırarak yeniden çalıştır.')

In [ ]:
# ── (OPSİYONEL) Toplu işleme ─────────────────────────────────────────────────

GOOD_INDICES = [2, 3, 6, 9, 12, 14, 15, 17, 19, 20, 21, 25, 26, 27, 29]
SAVE_DIR     = ('/content/drive/MyDrive/sutun_veriseti_1/restored'
                if IN_COLAB else './restored')
os.makedirs(SAVE_DIR, exist_ok=True)

damaged_files = [f for f in os.listdir(DAMAGED_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for idx in GOOD_INDICES:
    if idx >= len(damaged_files): continue
    path = os.path.join(DAMAGED_DIR, damaged_files[idx])
    try:
        res = restore_column(path, extra_bottom=0.50, bottom_gap_thresh=0.05)
        if res:
            bgr  = cv2.cvtColor(res['final'], cv2.COLOR_RGB2BGR)
            save = os.path.join(SAVE_DIR, f'restored_{idx:02d}_{damaged_files[idx]}')
            cv2.imwrite(save, bgr)
            print(f'  [{idx}] → {save}')
    except Exception as e:
        print(f'  [{idx}] Hata: {e}')

print('Toplu işleme tamamlandı.')